# 29 — Responsible & Robust NLP: PII, Bias, Perturbations and Safety Boundaries

**Learning objective.** Test text systems against privacy-sensitive inputs and simple distribution/robustness perturbations.

This notebook follows the track contract: concept → inspectable implementation → rendered result → failure modes → production implication.

## Mental model

**real-world input → privacy/robustness/governance controls → risk-aware system behavior**

Follow the information transformation first; treat the API as an implementation detail.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Redact **PII** before logging/model calls | sensitive literal information is removed | privacy risk drops but some task signal may disappear |
| Add perturbation tests | input variations are explored | brittle shortcuts become visible |
| Raise escalation threshold | more uncertain cases go to humans | automation falls while severe-error risk can drop |

> Write down what should move downstream before changing a control.

## Think before running the next cell

1. If a model is 95% accurate overall but fails badly on one critical slice, is it robust?
2. What information can logs leak even when predictions are correct?

### When to use
Use privacy, robustness and governance checks as part of the definition of correctness.

### When not to use / caution
Do not treat a checklist as proof of safety; risk depends on deployment context and consequences.

### Debugging lens
Test failure modes intentionally: PII, typos, long inputs, OOD text, code-switching, adversarial strings and sensitive slices.

In [1]:
from pathlib import Path
import re, random, math, json
import numpy as np
import pandas as pd
np.random.seed(42); random.seed(42)
print("Reproducibility seed: 42")

Reproducibility seed: 42


In [2]:
samples=[
 'Email alice@example.com or call +91 98765 43210',
 'Card ending 1234 was charged twice',
 'My name is Ravi and my email is ravi.k@example.org']
email_re=re.compile(r'[\w.+-]+@[\w.-]+\.\w+')
phone_re=re.compile(r'(?<!\d)(?:\+?91[ -]?)?[6-9]\d{4}[ -]?\d{5}(?!\d)')
def redact(s):
    return phone_re.sub('<PHONE>',email_re.sub('<EMAIL>',s))
for s in samples:
    print('RAW:',s); print('SAFE:',redact(s)); print()

RAW: Email alice@example.com or call +91 98765 43210
SAFE: Email <EMAIL> or call <PHONE>

RAW: Card ending 1234 was charged twice
SAFE: Card ending 1234 was charged twice

RAW: My name is Ravi and my email is ravi.k@example.org
SAFE: My name is Ravi and my email is <EMAIL>



In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
train=['great product','excellent product','bad product','terrible product']*8
y=['pos','pos','neg','neg']*8
m=Pipeline([('v',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5))),('c',LogisticRegression(random_state=42))]).fit(train,y)
tests=['great product','gr8 product','great   product','GREAT product','terrible product','terribl product']
print(pd.DataFrame({'input':tests,'prediction':m.predict(tests)}).to_string(index=False))

           input prediction
   great product        pos
     gr8 product        pos
 great   product        pos
   GREAT product        pos
terrible product        neg
 terribl product        neg


## Robustness and governance checklist
- **Privacy:** identify PII/PHI/secrets before logging or sending text to external services.
- **Bias:** define protected/sensitive slices appropriate to the jurisdiction and product; compare errors, not only aggregate accuracy.
- **Robustness:** test casing, whitespace, misspellings, code-switching, long inputs, adversarial strings and out-of-domain text.
- **Security:** treat retrieved/user text as untrusted data; generation systems need prompt-injection/tool authorization controls.
- **Human impact:** use abstention/escalation when model uncertainty or consequence severity makes automation unsafe.
- **Auditability:** log model/data versions and decisions without retaining unnecessary sensitive text.

---
## Production takeaways
- Treat decoding, privacy, robustness and task heads as explicit system design choices.
- Keep evaluation aligned with the actual task and deployment risk.